# 🧹 Notebook 2: Data Cleaning
**Project:** TalentSight — Employee Attrition Intelligence Platform  
**Author:** Saheri  
**Date:** May 2026  

## Objective
Act on every issue identified in `01_Data_Audit.ipynb` and produce a single 
cleaned dataset ready for EDA and feature engineering.

**Cleaning decisions made in this notebook:**
1. Drop zero-variance columns — `EmployeeCount`, `Over18`, `StandardHours`
2. Cast 9 ordinal columns to ordered `category` dtype

**What is NOT done here:**
- Encoding of `Attrition` target variable — handled in `05_Feature_Engineering.ipynb`
- One-hot or label encoding of nominal columns — handled in feature engineering
- Outlier removal — audit confirmed outliers are real business patterns, not errors
- No rows are dropped — all 1,470 records are retained

The output of this notebook is saved to `data/processed/cleaned_employee_attrition.csv`.

## 1. Setup & Data Loading
Load the raw IBM dataset from `data/raw/`. All cleaning operations are applied 
to a copy of this dataframe — the raw file is never modified.

In [1]:
#Import Necessaary Libraries
import os
import pandas as pd
import numpy as np

In [2]:
#Load the dataset
BASE_DIR = os.getcwd()
data = pd.read_csv(os.path.join(BASE_DIR, '..', 'data', 'raw', 'WA_Fn-UseC_-HR-Employee-Attrition.csv'))

### Confirmed: Raw data loaded successfully
1,470 rows × 35 columns loaded from raw source. This matches the audit baseline.
No transformations applied yet.

## 2. Drop Constant Columns
Three columns were flagged in the audit as zero-variance HRIS export artifacts:

| Column | Unique Values | Reason for Dropping |
|---|---|---|
| `EmployeeCount` | 1 (always = 1) | No predictive value |
| `Over18` | 1 (always = 'Y') | No predictive value |
| `StandardHours` | 1 (always = 80) | No predictive value |

Dropping these reduces noise and keeps the feature space clean.

In [3]:
#Drop the constant columns
Constant_Columns = [col for col in data.columns if data[col].nunique()==1]
data.drop(columns=Constant_Columns, inplace=True)

In [4]:
#Print the constant columns that were dropped
print("Constant columns that were dropped:", Constant_Columns)
#Check the shape of the dataset after dropping the constant columns
print("Shape of the dataset after dropping constant columns:", data.shape)

Constant columns that were dropped: ['EmployeeCount', 'Over18', 'StandardHours']
Shape of the dataset after dropping constant columns: (1470, 32)


## 3. Verify Target Variable
`Attrition` remains as Yes/No strings at this stage — encoding to binary 0/1 
is handled in `05_Feature_Engineering.ipynb` immediately before modelling.

This separation is intentional: the cleaned CSV is used by both the EDA notebook 
(which benefits from readable Yes/No labels in charts) and the feature engineering 
notebook (which needs binary integers for the model).

In [5]:
#Verify the encoding of the target variable
print("Unique values in 'Attrition' column after encoding:", data['Attrition'].unique())    

Unique values in 'Attrition' column after encoding: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


## 4. Cast Ordinal Columns to Ordered Category Dtype
9 columns are ordinal scales encoded as `int64`. While tree-based models like 
XGBoost can handle these natively, casting them to pandas `category` dtype with 
explicit ordering:

- Documents our understanding that these are scales, not continuous numbers
- Prevents accidental treatment as continuous in any linear models
- Makes the data contract explicit for anyone reading the codebase

**Columns cast:**
`Education` (1–5), `EnvironmentSatisfaction` (1–4), `JobInvolvement` (1–4),
`JobLevel` (1–5), `JobSatisfaction` (1–4), `PerformanceRating` (1–4),
`RelationshipSatisfaction` (1–4), `StockOptionLevel` (0–3), `WorkLifeBalance` (1–4)

**Columns intentionally left as `object` for now:**
`BusinessTravel`, `Department`, `EducationField`, `Gender`, `JobRole`, 
`MaritalStatus`, `OverTime` — these are nominal categoricals that will be 
one-hot encoded in the feature engineering notebook via ColumnTransformer.

In [6]:
# Cast ordinal columns to ordered categorical dtype to preserve scale hierarchy
ordinal_cols = [
    'Education', 'EnvironmentSatisfaction', 'JobInvolvement',
    'JobLevel', 'JobSatisfaction', 'PerformanceRating',
    'RelationshipSatisfaction', 'StockOptionLevel', 'WorkLifeBalance'
]

for col in ordinal_cols:
    data[col] = pd.Categorical(data[col], ordered=True)
#Check the data types of the columns after encoding and casting
print("Data types of the columns after encoding and casting:")
print(data.dtypes)

Data types of the columns after encoding and casting:
Age                            int64
Attrition                        str
BusinessTravel                   str
DailyRate                      int64
Department                       str
DistanceFromHome               int64
Education                   category
EducationField                   str
EmployeeNumber                 int64
EnvironmentSatisfaction     category
Gender                           str
HourlyRate                     int64
JobInvolvement              category
JobLevel                    category
JobRole                          str
JobSatisfaction             category
MaritalStatus                    str
MonthlyIncome                  int64
MonthlyRate                    int64
NumCompaniesWorked             int64
OverTime                         str
PercentSalaryHike              int64
PerformanceRating           category
RelationshipSatisfaction    category
StockOptionLevel            category
TotalWorkingYears    

### Confirmed: Ordinal columns correctly typed
All 9 ordinal columns now show `category` dtype. The 7 nominal string columns 
remain as `object` — correct, pending encoding in feature engineering.

**Important note for subsequent notebooks:** CSV format does not preserve 
`category` dtype. When loading `cleaned_employee_attrition.csv` in later 
notebooks, the ordinal casting step must be reapplied before modelling.

In [7]:
#Save the cleaned dataset
data.to_csv(os.path.join(BASE_DIR, '..', 'data', 'processed', 'cleaned_employee_attrition.csv'), index=False)

## 5. Save Cleaned Dataset
The cleaned dataframe is exported to `data/processed/cleaned_employee_attrition.csv`.

**The raw file `data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv` is never modified.**
This separation ensures full reproducibility — the cleaning notebook can always 
be rerun from scratch against the original source.

In [8]:
#Verify if the cleaned dataset is saved correctly
print("Cleaned dataset saved successfully!")
print(f"Shape: {data.shape}")
print(f"Columns: {data.columns.tolist()}")

Cleaned dataset saved successfully!
Shape: (1470, 32)
Columns: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## Cleaning Complete — Summary

| Step | Action | Result |
|---|---|---|
| Drop constants | Removed 3 zero-variance columns | 35 → 32 columns |
| Verify target | Attrition confirmed as Yes/No strings | Encoding deferred to feature engineering |
| Cast ordinals | 9 columns → ordered `category` | Dtype contract documented |
| Save output | `data/processed/cleaned_employee_attrition.csv` | ✅ |

---

**Next:** `03_EDA.ipynb` — answer business questions with data, not just plots.